# Quantum Fourier Transform

Build the QFT from Hadamards and controlled phase rotations, verify it against a direct numpy computation, and recover the input with the inverse QFT.

**Objectives:**
- Construct the QFT circuit explicitly with `.h`, `.cphaseshift`, and `.swap`
- Verify `QFT|j>` matches the exact phase-spread formula for several inputs
- Pin down the index, sign, and normalization conventions against `numpy.fft`
- Use `.adjoint()` to build the inverse QFT and recover the original state
- See the QFT read periodicity as sharp spikes in a measurement histogram

**Reference:** See [`../GUIDE.md`](../GUIDE.md) for concept explanations and context.
<!-- browser-runnable -->

In [ ]:
# Setup: Run this cell first
# Requires: pip install -e '.[dev]' from the project root (see `make setup`)

from braket.circuits import Circuit
from braket.devices import LocalSimulator
import numpy as np
import matplotlib.pyplot as plt

from lib.circuits import bell_pair, ghz_state, qft_circuit
from lib.utils.results import parse_counts
from lib.utils.visualization import plot_histogram

device = LocalSimulator()

## 1. What the QFT does

The Quantum Fourier Transform is the quantum Discrete Fourier Transform. On $n$ qubits ($N = 2^n$ basis states) it maps a computational basis state $|j>$ to an even spread of phases:

$$
\text{QFT}\,|j\rangle \;=\; \frac{1}{\sqrt{N}} \sum_{k=0}^{N-1} e^{\,2\pi i\, j k / N}\,|k\rangle .
$$

Every output basis state $|k>$ gets the same magnitude $1/\sqrt{N}$; the information lives entirely in the **phases**, which wind around the unit circle at a rate set by $j$. That is the whole idea: the QFT converts "which basis state" into "how fast the phase winds," so a state that repeats with some period turns into sharp spikes at the matching frequency.

The famous part is the cost: built from Hadamards and controlled phase rotations, the QFT needs only $O(n^2)$ gates, exponentially fewer than the classical FFT's $O(n\,2^n)$ on the amplitude vector.

**Convention reminder (critical for every assert below):** qcsim is *big-endian* — qubit 0 is the most-significant (leftmost) bit of every basis index and every measured bitstring. So the basis index $j$ has its highest-order bit on qubit 0.

## 2. Build the QFT from scratch

The recipe, qubit by qubit, mirrors `lib.circuits.qft_circuit` but we build it inline so every gate is visible:

1. For each qubit `i`: apply a Hadamard `h(i)`.
2. Then, for each later qubit `j` (with `j > i`): apply a controlled phase rotation `cphaseshift(j, i, pi / 2**(j-i))`. The phase shrinks by half for each extra qubit of separation.
3. After all the rotations, reverse the qubit order with `swap(i, n-1-i)` for `i` in the first half. The phase-rotation network produces the output bits in reversed order, and the final swaps put them back.

Note that `cphaseshift` applies a diagonal phase `e^{i*angle}` to the `|11>` component of its two qubits, so it is symmetric in its first two arguments — the `(j, i)` order matches the library and the resulting phase is identical either way.

In [ ]:
def build_qft(n):
    """Explicit QFT on n qubits (qubits 0..n-1), big-endian."""
    c = Circuit()
    for i in range(n):
        c.h(i)
        for j in range(i + 1, n):
            c.cphaseshift(j, i, np.pi / 2 ** (j - i))
    for i in range(n // 2):
        c.swap(i, n - 1 - i)
    return c


qft3 = build_qft(3)
print(qft3)
print(f"\nqubits: {qft3.qubit_count}")
print(f"gate count: {sum(1 for _ in qft3.instructions)}  (3 H + 3 cphaseshift + 1 swap)")
print(f"depth: {qft3.depth}")

# Sanity check: our explicit build matches the shared library helper exactly,
# gate for gate, by comparing the full state vectors on the |000> input.
assert np.allclose(build_qft(3).state_vector(), qft_circuit(3).state_vector(), atol=1e-9)
print("\nVerified: build_qft(3) reproduces lib.circuits.qft_circuit(3) exactly.")

## 3. Prepare a basis-state input (big-endian)

To feed the QFT a computational basis state $|j>$, we put each qubit into $|0>$ or $|1>$ to spell out $j$ in binary. Because qubit 0 is the most-significant bit, qubit $q$ carries bit $(j >> (n-1-q))\,\&\,1$. We apply an `x` to exactly the qubits whose bit is 1.

Getting this index mapping right is what makes the numpy comparison line up, so we verify it directly: prepare $|j>$ with no QFT, and the resulting state vector must be a single 1 at index $j$.

In [ ]:
def prep_basis(n, j):
    """Circuit preparing the big-endian computational basis state |j> on n qubits."""
    c = Circuit()
    for q in range(n):
        if (j >> (n - 1 - q)) & 1:
            c.x(q)
        else:
            c.i(q)  # touch every qubit so the register spans all n (Braket compacts unused ones)
    return c


# Verify the index convention: |j> must be a single amplitude 1.0 at index j.
for n, j in [(3, 1), (3, 5), (4, 5)]:
    sv = prep_basis(n, j).state_vector()
    expected = np.zeros(2 ** n, dtype=complex)
    expected[j] = 1.0
    assert np.allclose(sv, expected, atol=1e-9), (n, j)
    bits = format(j, f"0{n}b")
    print(f"n={n}: |{j}> prepared correctly  (qubit0..{n-1} bits = {bits})")
print("\nBig-endian index convention confirmed.")

## 4. Verify `QFT|j>` against the exact formula

Now the headline claim. For an input basis state $|j>$, the QFT output amplitude at index $k$ must be exactly

$$
\text{amp}[k] = \frac{1}{\sqrt{N}}\, e^{\,2\pi i\, j k / N}.
$$

We build that vector directly in numpy and compare it to `circuit.state_vector()`. We use the exact state vector (not sampling) because the claim is about amplitudes, including their phases, which sampling cannot see. We check two different inputs on $n=3$ and two on $n=4$.

In [ ]:
def qft_basis_expected(n, j):
    """Exact QFT output vector for input |j>: amp[k] = exp(+2pi i j k / N) / sqrt(N)."""
    N = 2 ** n
    k = np.arange(N)
    return np.exp(2j * np.pi * j * k / N) / np.sqrt(N)


for n, j in [(3, 1), (3, 5), (4, 1), (4, 5)]:
    circ = prep_basis(n, j)
    circ.add_circuit(build_qft(n))
    sv = circ.state_vector()
    expected = qft_basis_expected(n, j)
    assert np.allclose(sv, expected, atol=1e-9), (n, j)
    # All output magnitudes are equal (1/sqrt(N)); info is in the phases.
    mags = np.abs(sv)
    print(f"n={n}, j={j}: matches formula. |amp| all = {mags[0]:.4f} = 1/sqrt({2**n}) = {1/np.sqrt(2**n):.4f}")

print("\nVerified: QFT|j> equals the exact phase-spread formula for every input tested.")

## 5. Pin down the relationship to `numpy.fft`

The QFT is *a* Fourier transform, but `numpy.fft` uses different conventions, and mixing them up is the classic source of off-by-a-sign / off-by-a-factor bugs. The two differences:

- **Sign of the exponent.** `np.fft.fft(a)[k] = sum_j a_j e^{-2\pi i j k / N}` uses a *minus* sign. The QFT uses a *plus* sign.
- **Normalization.** `np.fft.fft` has no $1/\sqrt{N}$ out front; the QFT divides by $\sqrt{N}$.

The plus-sign-with-a-factor combination is exactly `np.fft.ifft` (inverse FFT) times $\sqrt{N}$, since

$$
\text{ifft}(a)[k] = \frac{1}{N}\sum_j a_j\, e^{+2\pi i j k / N}
\;\Longrightarrow\;
\sqrt{N}\,\text{ifft}(a)[k] = \frac{1}{\sqrt{N}}\sum_j a_j\, e^{+2\pi i j k / N} = \text{QFT}(a)[k].
$$

So for *any* input amplitude vector $a$ on the register, `QFT(a) == sqrt(N) * np.fft.ifft(a)`. We verify it both for a basis-state input and for a genuine superposition input.

In [ ]:
# (a) Basis-state input: a is a delta at j.
n, j = 3, 1
N = 2 ** n
circ = prep_basis(n, j)
circ.add_circuit(build_qft(n))
sv = circ.state_vector()

a = np.zeros(N, dtype=complex)
a[j] = 1.0
numpy_qft = np.sqrt(N) * np.fft.ifft(a)
assert np.allclose(sv, numpy_qft, atol=1e-9)
print("(a) delta input: QFT state == sqrt(N) * np.fft.ifft(a)")

# (b) Superposition input: H on qubit 0 gives (|000> + |100>)/sqrt(2),
#     i.e. amplitude 1/sqrt(2) at indices 0 and 4 (big-endian).
n = 3
N = 2 ** n
circ = Circuit()
circ.h(0).i(1).i(2)             # superposition on qubit 0; span all 3 qubits
input_state = circ.state_vector()  # the register amplitudes BEFORE the QFT (length 8)
circ.add_circuit(build_qft(n))
sv = circ.state_vector()

numpy_qft = np.sqrt(N) * np.fft.ifft(input_state)
assert np.allclose(sv, numpy_qft, atol=1e-9)
print("(b) superposition input: QFT state == sqrt(N) * np.fft.ifft(input_state)")
print("\nConfirmed: np.fft.fft uses exp(-...) with no 1/sqrt(N); the QFT is the")
print("plus-sign, sqrt(N)-normalized partner == sqrt(N) * np.fft.ifft.")

## 6. The inverse QFT recovers the input

The QFT is unitary, so it has an exact inverse. `Circuit.adjoint()` returns a new circuit with the gates reversed and each gate replaced by its conjugate transpose — exactly $U^{\dagger}$. Applying the QFT and then its adjoint must return the register to wherever it started.

We prepare $|j>$, apply `qft`, then `qft.adjoint()`, and assert the final state vector is the single-amplitude $|j>$ again.

In [ ]:
n, j = 4, 5
qft = build_qft(n)

circ = prep_basis(n, j)
circ.add_circuit(qft)             # forward QFT
circ.add_circuit(qft.adjoint())   # inverse QFT
sv = circ.state_vector()

expected = np.zeros(2 ** n, dtype=complex)
expected[j] = 1.0
assert np.allclose(sv, expected, atol=1e-9)
print(f"QFT then inverse-QFT on |{j}> returns exactly |{j}>.")

# It is a true inverse for ALL inputs, not just one: check it on a superposition too.
n = 3
qft = build_qft(n)
circ = Circuit()
circ.h(0).h(1).i(2)               # arbitrary superposition; span all 3 qubits
before = circ.state_vector()
circ.add_circuit(qft)
circ.add_circuit(qft.adjoint())
after = circ.state_vector()
assert np.allclose(before, after, atol=1e-9)
print("Inverse-QFT also recovers a superposition input exactly (adjoint is U-dagger).")

## 7. Now measure it: the QFT reads periodicity

The asserts above used exact amplitudes. Here we do the pedagogical "now run it on the simulator and look" step, which is sampled and therefore approximate.

The QFT's signature application is reading a **period**. We prepare a periodic comb on $n=4$ qubits — an equal superposition of every state that is a multiple of a period $r$ — and apply the QFT. Constructive interference should pile probability onto multiples of $N/r$, turning a comb in the position basis into a comb in the frequency basis.

With $N = 16$ and period $r = 4$, the input is the equal superposition of $|0>, |4>, |8>, |12>$. We expect spikes at frequencies $0, 4, 8, 12$ (the multiples of $N/r = 4$).

In [ ]:
n = 4
N = 2 ** n
r = 4  # period

# Build the periodic-comb input as a small sub-circuit per basis state would be
# awkward; instead we verify the state vector directly, then sample the QFT output.
# Equal superposition over {0, 4, 8, 12}: indices that are multiples of r.
# Big-endian index 4 = '0100', 8 = '1000', 12 = '1100'.
# We can prepare {0,4,8,12} = { x*4 : x in 0..3 } by putting qubits 0 and 1 into
# superposition (they form the top 2 bits) and leaving qubits 2,3 in |0>.
comb = Circuit()
comb.h(0).h(1).i(2).i(3)   # top two bits range over 00,01,10,11 -> indices 0,4,8,12; span all 4 qubits

comb_state = comb.state_vector()
support = np.flatnonzero(np.abs(comb_state) > 1e-9)
print(f"input comb support (indices): {support.tolist()}  (multiples of N/r = {N//r})")
assert set(support.tolist()) == {0, 4, 8, 12}

# Apply the QFT and sample.
circ = Circuit()
circ.h(0).h(1)
circ.add_circuit(build_qft(n))

result = device.run(circ, shots=2000).result()
counts = result.measurement_counts
probs = {bs: c / 2000 for bs, c in counts.items()}
top = sorted(probs.items(), key=lambda kv: -kv[1])[:6]
print("\nTop QFT output bitstrings (big-endian, qubit0 leftmost):")
for bs, p in top:
    print(f"  {bs}  index={int(bs, 2):2d}  prob~{p:.3f}")

plot_histogram(counts, title=f"QFT of a period-{r} comb on n={n} qubits")
plt.show()

The probability concentrates on the frequencies that are multiples of $N/r = 4$ — the QFT turned a period-4 comb into spikes at $0, 4, 8, 12$. Reading those spikes back out is exactly how period-finding (and Shor's algorithm) recovers the hidden period. (The exact frequencies were guaranteed by the amplitude asserts above; the histogram just shows the same fact through sampling, so small statistical wobble in the bar heights is expected.)

## Exercises

Work through these by editing the scaffolds. No solutions are given — verify your own answers with `state_vector()` and an `assert`, the same way the sections above did.

1. **Different size.** Build the QFT on $n=2$ qubits by hand (2 Hadamards, 1 `cphaseshift`, 1 swap) and confirm it matches `build_qft(2)`.
2. **A new input.** Pick an input $j$ you have not tested (say $j=3$ on $n=3$) and assert `QFT|j>` matches `qft_basis_expected(3, 3)`.
3. **Inverse-only.** Prepare a QFT output state, then apply only the inverse QFT and recover the original $|j>$ — read off $j$ from the single nonzero amplitude.

In [ ]:
# Exercise 1: build QFT(2) by hand and compare to build_qft(2)
# c = Circuit()
# c.h(0)
# c.cphaseshift(1, 0, np.pi / 2 ** 1)   # the single controlled phase
# c.swap(0, 1)
# assert np.allclose(c.state_vector(), build_qft(2).state_vector(), atol=1e-9)
# print("QFT(2) by hand matches the helper.")

# Exercise 2: verify QFT|3> on n=3 against the exact formula
# n, j = 3, 3
# circ = prep_basis(n, j)
# circ.add_circuit(build_qft(n))
# assert np.allclose(circ.state_vector(), qft_basis_expected(n, j), atol=1e-9)
# print("QFT|3> matches the formula.")

In [ ]:
# Exercise 3: round-trip through the inverse QFT and read off the input index
# n, j = 3, 6
# qft = build_qft(n)
# circ = prep_basis(n, j)
# circ.add_circuit(qft)            # forward
# circ.add_circuit(qft.adjoint())  # inverse
# sv = circ.state_vector()
# recovered = int(np.argmax(np.abs(sv)))
# assert recovered == j
# print(f"Recovered input index = {recovered}")

## Summary

- The QFT maps $|j>$ to an even spread of equal-magnitude amplitudes whose **phases** wind at a rate set by $j$: $\text{QFT}|j> = \frac{1}{\sqrt{N}}\sum_k e^{2\pi i j k/N}|k>$.
- It is built from $O(n^2)$ gates: a Hadamard per qubit, controlled `cphaseshift` rotations that shrink by half with qubit distance, and a final layer of swaps to undo the bit reversal.
- Against `numpy.fft`: `np.fft.fft` uses the **minus** sign and no normalization, so the QFT is its plus-sign, $\sqrt{N}$-normalized partner — exactly `sqrt(N) * np.fft.ifft`.
- `Circuit.adjoint()` gives the exact **inverse QFT**; QFT then inverse-QFT is the identity on any input.
- Fed a periodic state, the QFT concentrates probability at the matching frequencies — the engine behind quantum period-finding.

**Next:** [`04-qpe.ipynb`](04-qpe.ipynb) — Quantum Phase Estimation points the inverse QFT at a unitary's eigenphase to read it out in binary.